# 01 — Understanding Token Limits — Practical Examples

**Covers**: tiktoken counting, message token math, tool token costs, cost calculator, usage monitoring

In [ ]:
import os, json
import tiktoken
from openai import OpenAI
from dotenv import load_dotenv
from rich import print as rprint

load_dotenv()
client = OpenAI()
MODEL = 'gpt-4o-mini'
enc = tiktoken.encoding_for_model(MODEL)
print('✅ Setup complete')

## 1. Token Anatomy — See Exactly How Text Is Tokenized

In [ ]:
def show_tokenization(text: str):
    """Visualize how text is split into tokens."""
    token_ids = enc.encode(text)
    token_strings = [enc.decode([t]) for t in token_ids]
    print(f"Text:         {text!r}")
    print(f"Token count:  {len(token_ids)}")
    print(f"Token IDs:    {token_ids}")
    print(f"Tokens:       {token_strings}")
    print(f"Chars/token:  {len(text)/len(token_ids):.1f}")
    print()

show_tokenization("Hello, world!")
show_tokenization("Context window management is critical for agentic AI systems.")
show_tokenization("gpt-4o-mini")
show_tokenization("The quick brown fox jumps over the lazy dog")

## 2. Counting Tokens in a Messages Array

In [ ]:
def count_message_tokens(messages: list[dict], model: str = MODEL) -> int:
    """Count total tokens in messages array (matches OpenAI's counting)."""
    enc_local = tiktoken.encoding_for_model(model)
    tokens_per_message = 3   # Every message has role + content delimiters
    tokens_per_name = 1      # If 'name' field is present
    total = 3                # Reply priming tokens
    for msg in messages:
        total += tokens_per_message
        for key, value in msg.items():
            total += len(enc_local.encode(str(value)))
            if key == 'name':
                total += tokens_per_name
    return total

# Test with a realistic agent conversation
conversation = [
    {"role": "system",    "content": "You are a helpful research assistant. Use tools to find accurate information."},
    {"role": "user",      "content": "What is the current population of Tokyo?"},
    {"role": "assistant", "content": "I'll search for the current population of Tokyo."},
    {"role": "user",      "content": "Also compare it to New York City."},
    {"role": "assistant", "content": "Tokyo has approximately 14 million in the city and 37 million in the metro area. New York City has about 8.3 million city residents and 20 million metro."},
]

for i in range(1, len(conversation) + 1):
    subset = conversation[:i]
    tokens = count_message_tokens(subset)
    last = conversation[i-1]
    print(f"After {i} msg(s): {tokens:,} tokens | Last role: {last['role']}")

## 3. Tool Definition Token Cost

In [ ]:
def count_tool_tokens(tools: list[dict], model: str = MODEL) -> int:
    enc_local = tiktoken.encoding_for_model(model)
    schema_str = json.dumps(tools)
    return len(enc_local.encode(schema_str)) + 15  # +15 for OpenAI overhead

TOOLS = [
    {"type": "function", "function": {
        "name": "web_search",
        "description": "Search the web for current information.",
        "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}
    }},
    {"type": "function", "function": {
        "name": "get_weather",
        "description": "Get current weather for a city.",
        "parameters": {"type": "object", "properties": {"city": {"type": "string"}, "units": {"type": "string", "enum": ["celsius", "fahrenheit"]}}, "required": ["city"]}
    }},
    {"type": "function", "function": {
        "name": "run_code",
        "description": "Execute Python code in a sandbox. Returns stdout and any errors.",
        "parameters": {"type": "object", "properties": {"code": {"type": "string"}, "timeout_seconds": {"type": "integer"}}, "required": ["code"]}
    }}
]

for n in range(1, len(TOOLS) + 1):
    t = count_tool_tokens(TOOLS[:n])
    print(f"{n} tool(s): ~{t} tokens overhead per API call")

## 4. Real-Time Token Usage from API Response

In [ ]:
# Every API response includes exact token usage
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system",  "content": "You are a helpful assistant."},
        {"role": "user",    "content": "Explain context window management in 2 sentences."}
    ],
    max_tokens=200
)

usage = response.usage
print(f"Prompt tokens:     {usage.prompt_tokens:,}")
print(f"Completion tokens: {usage.completion_tokens:,}")
print(f"Total tokens:      {usage.total_tokens:,}")
print(f"\nResponse: {response.choices[0].message.content}")

# Calculate actual cost
INPUT_PRICE  = 0.150 / 1_000_000   # gpt-4o-mini input
OUTPUT_PRICE = 0.600 / 1_000_000   # gpt-4o-mini output
cost = (usage.prompt_tokens * INPUT_PRICE) + (usage.completion_tokens * OUTPUT_PRICE)
print(f"\nCost of this call: ${cost:.6f}")

## 5. Cost Calculator — Agent Run vs Single Call

In [ ]:
PRICING = {
    "gpt-4o-mini":          {"input": 0.150/1e6, "output": 0.600/1e6},
    "gpt-4o":               {"input": 5.00/1e6,  "output": 15.00/1e6},
    "claude-3-haiku":       {"input": 0.25/1e6,  "output": 1.25/1e6},
    "claude-3-5-sonnet":    {"input": 3.00/1e6,  "output": 15.00/1e6},
}

def estimate_agent_run(in_tokens_per_step, out_tokens_per_step, steps, model='gpt-4o-mini'):
    p = PRICING.get(model, PRICING['gpt-4o-mini'])
    per_step = in_tokens_per_step * p['input'] + out_tokens_per_step * p['output']
    total = per_step * steps
    return {'per_step': round(per_step, 6), 'total': round(total, 4),
            'per_1k_runs': round(total * 1000, 2), 'per_million_runs': round(total * 1e6, 0)}

print(f"{'Model':<22} {'$/step':>8} {'Total (10s)':>12} {'Per 1k runs':>12}")
print('-' * 60)
for model in PRICING:
    r = estimate_agent_run(in_tokens_per_step=2000, out_tokens_per_step=500, steps=10, model=model)
    print(f"{model:<22} ${r['per_step']:>7.5f} ${r['total']:>11.4f} ${r['per_1k_runs']:>11.2f}")

## 6. Context Growth Visualizer — See the Problem

In [ ]:
# Simulate growing context in an agent loop (no API calls needed)
import tiktoken

def simulate_context_growth(num_turns: int = 15, tokens_per_turn: int = 300):
    """Show how token count grows as agent runs."""
    enc_local = tiktoken.encoding_for_model(MODEL)
    system_tokens = 150
    context_limit = 128_000
    safe_limit = context_limit - 4096  # Reserve for output
    
    total_tokens = system_tokens
    print(f"{'Turn':<6} {'Total Tokens':>13} {'% Used':>8} {'Status'}")
    print('-' * 50)
    for turn in range(1, num_turns + 1):
        total_tokens += tokens_per_turn * 2  # user + assistant each turn
        pct = total_tokens / context_limit * 100
        status = '🟢 OK' if total_tokens < safe_limit * 0.5 else ('🟡 Watch' if total_tokens < safe_limit * 0.8 else '🔴 Critical')
        print(f"{turn:<6} {total_tokens:>13,} {pct:>7.1f}% {status}")
        if total_tokens > safe_limit:
            print(f"  ⚠️  Context limit approaching after {turn} turns!")
            break

simulate_context_growth(num_turns=20, tokens_per_turn=500)

## 📌 Summary

- **tiktoken**: count tokens before sending — `enc.encode(text)` → token IDs
- **Message overhead**: 3 tokens per message + role + content 
- **Tool costs**: each tool definition adds ~80-150 tokens per call
- **`response.usage`**: exact token counts from the API after each call
- **Cost math**: (prompt_tokens × input_price) + (completion_tokens × output_price)
- **Context grows** every turn — without management it eventually hits the limit